# Fuzzy LPQ / LPQ6 / MP-LPQ with Fisher Vector Embedding
### Reference implementation for *"Fuzzy Local Phase Quantization with Fisher Vector Embedding for Blur-Insensitive Texture Classification"*, manuscript to be submitted to Expert Systems With Applications

This notebook implements, end-to-end, the ideas of the paper:

1. **STFT phase coefficient extraction** (Algorithm 1)
2. **Crisp LPQ, LPQ6, MP-LPQ** histogram descriptors (Algorithm 2)
3. **Fuzzy quantization**: sigmoid ($K=2$) and Gaussian RBF ($K>2$) membership functions (Eqs. 3-4)
4. **Adaptive fuzzy partition learning**, EM-style gradient-on-center procedure, general $K \ge 2$ (Algorithm 3)
5. **Fuzzy soft-histogram extraction**: exact joint-code histogram via the Kronecker-product trick for $K=2$ (Algorithms 4-5), tractable marginal histogram for $K>2$
6. **Blur degradation bank** matching MATLAB's `fspecial` (Gaussian $\sigma\in\{1,2,3\}$ / motion $\lambda\in\{8,9\}$ / disk $\gamma\in\{2,3\}$)
7. **Fisher Vector encoding** over a diagonal-covariance GMM (Algorithms 6-7), applied to **both** the crisp (hard 0/1) and the fuzzy (soft) local descriptors, for each of LPQ / LPQ6 / MP-LPQ
8. **End-to-end classification pipeline** (Algorithm 8): dataset loading &rarr; descriptor extraction &rarr; SVM training/evaluation, with reusable fit/evaluate helpers so a model is trained once and tested across every blur condition
9. A **synthetic-texture demo** reproducing the paper's comparison table for all 12 method combinations (LPQ / LPQ6 / MP-LPQ &times; crisp\_hist / crisp\_fv / fuzzy\_hist / fuzzy\_fv), plus a dedicated $K>2$ ablation and instructions to plug in KTH-TIPS / Outex / Kylberg from Google Drive
10. **5-fold stratified cross-validation** across all methods, for a statistically robust comparison

Runs top-to-bottom on a standard Colab CPU runtime (no GPU required); everything is pure NumPy/SciPy/scikit-learn.


## 0. Setup

In [41]:
# Install required dependencies
!pip install -q scikit-image scikit-learn scipy pillow numpy


In [40]:
# Numerical computing
import numpy as np

# Fast Fourier transform-based convolution
from scipy.signal import fftconvolve

# Gaussian Mixture Models for clustering/density estimation
from sklearn.mixture import GaussianMixture

# Support Vector Machine classifier
from sklearn.svm import SVC

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report

# Plotting utilities
import matplotlib.pyplot as plt

# Ensure reproducible results across runs
np.random.seed(0)


In [ ]:
# Optional: mount Google Drive if you want to load KTH-TIPS / Outex / Kylberg
# from a folder structured as:  <root>/<class_name>/<image files>
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# @title Data Loading
# Mount point path for the benchmark database on Google Drive (change as needed)
!mkdir -p /content/Data/yale_db/
# Copy the database from Google Drive to the local runtime environment
!cp -r "/content/drive/MyDrive/FuzzyLPQPlus/Data/yale_db" /content/Data/
# List the copied database directory contents
!dir /content/Data/yale_db/


## 1. STFT phase coefficient extraction — Algorithm 1

For each 2-D frequency point $\mathbf{u}_i = a\,(i,j)$ with $a = 1/M$, we compute the local
short-term Fourier transform via a 2-D convolution with a complex exponential kernel
(the separable 1-D formulation from the paper is mathematically equivalent; the direct
2-D form below is used here for clarity). Real and imaginary parts of each frequency
point give two scalar coefficient maps.


In [45]:
def stft_coefficients(image, M=15, freq_points=((1,0),(0,1),(1,1),(1,-1))):
    """Return an array of shape (n=2*len(freq_points), H, W) of STFT phase coefficients."""
    image = image.astype(np.float64)
    a = 1.0 / M
    r = M // 2
    ys = np.arange(-r, r + 1)
    Y1, Y2 = np.meshgrid(ys, ys, indexing='ij')
    coeffs = []
    for (i, j) in freq_points:
        u1, u2 = a * i, a * j
        kernel = np.exp(-1j * 2 * np.pi * (u1 * Y1 + u2 * Y2))
        F = fftconvolve(image, kernel, mode='same')
        coeffs.append(F.real)
        coeffs.append(F.imag)
    return np.stack(coeffs, axis=0)


## 2. Crisp LPQ / LPQ6 / MP-LPQ — Algorithm 2

* **LPQ**: 4 frequency points, $M=17$ &rarr; 8 coefficients &rarr; 256-bin histogram.
* **LPQ6**: 6 frequency points (added second ring), larger window $M=19$ &rarr; 12 coefficients.
* **MP-LPQ**: $P=3$ independent frequency-point configurations, histograms concatenated.

Histograms are computed on a spatial grid of cells (default $4\times4$) and concatenated,
as is standard practice for locally-aggregated texture descriptors.


In [46]:
LPQ_CONFIG = dict(M=17, freq_points=((1, 0), (0, 1), (1, 1), (1, -1)))
LPQ6_CONFIG = dict(M=19, freq_points=((1, 0), (0, 1), (1, 1), (1, -1), (2, 0), (0, 2)))
MP_LPQ_CONFIGS = [
    dict(M=15, freq_points=((1, 0), (0, 1), (1, 1), (1, -1))),
    dict(M=17, freq_points=((1, 0), (0, 1), (1, 1), (1, -1))),
    dict(M=19, freq_points=((1, 0), (0, 1), (1, 1), (1, -1), (2, 0), (0, 2))),
]

CONFIG_MAP = {"LPQ": [LPQ_CONFIG], "LPQ6": [LPQ6_CONFIG], "MP-LPQ": MP_LPQ_CONFIGS}


def crisp_code_map(coeffs):
    bits = (coeffs >= 0).astype(np.uint32)
    n = bits.shape[0]
    weights = (2 ** np.arange(n)).reshape(n, 1, 1)
    return (bits * weights).sum(axis=0)


def _grid_slices(H, W, grid):
    gy, gx = grid
    ys = np.linspace(0, H, gy + 1).astype(int)
    xs = np.linspace(0, W, gx + 1).astype(int)
    for a_ in range(gy):
        for b_ in range(gx):
            yield slice(ys[a_], ys[a_ + 1]), slice(xs[b_], xs[b_ + 1])


def crisp_histogram(code_map, n_bits, grid=(4, 4)):
    H, W = code_map.shape
    n_codes = 2 ** n_bits
    hists = []
    for sy, sx in _grid_slices(H, W, grid):
        cell = code_map[sy, sx].ravel()
        h = np.bincount(cell, minlength=n_codes).astype(np.float64)
        h = h / (h.sum() + 1e-12)
        hists.append(h)
    return np.concatenate(hists)


def extract_crisp_descriptor(image, configs, grid=(4, 4)):
    parts = []
    for cfg in configs:
        coeffs = stft_coefficients(image, **cfg)
        code = crisp_code_map(coeffs)
        parts.append(crisp_histogram(code, n_bits=coeffs.shape[0], grid=grid))
    return np.concatenate(parts)


def crisp_local_descriptor(coeffs):
    """Per-pixel hard (0/1) bit vector, shape (H*W, n) -- the crisp counterpart of
    fuzzy_local_descriptor_binary, used as the local feature fed to Fisher Vector
    encoding for the crisp descriptor (i.e. the tau -> 0 limit of the fuzzy local
    descriptor)."""
    bits = (coeffs >= 0).astype(np.float64)
    n, H, W = bits.shape
    return bits.reshape(n, -1).T


def extract_crisp_descriptor_with_local(image, configs, grid=(4, 4)):
    """Returns (histogram_descriptor, local_descriptors) for the crisp family, so the
    same image pass can feed either the plain histogram (mode='crisp_hist') or a
    Fisher Vector built over the hard bit vectors (mode='crisp_fv')."""
    hist_parts, local_parts = [], []
    for cfg in configs:
        coeffs = stft_coefficients(image, **cfg)
        code = crisp_code_map(coeffs)
        hist_parts.append(crisp_histogram(code, n_bits=coeffs.shape[0], grid=grid))
        local_parts.append(crisp_local_descriptor(coeffs))
    return np.concatenate(hist_parts), np.concatenate(local_parts, axis=1)


## 3. Fuzzy membership functions — Eqs. (3)-(4)

* `SigmoidFuzzyQuantizer`: binary ($K=2$) case, Eq. (3). As $\tau_k \to 0$ it recovers the
  crisp sign quantization exactly.
* `GaussianRBFPartition`: general $K$-level case, Eq. (4), a normalized Gaussian RBF
  partition of unity (numerically stabilized with a log-sum-exp shift).


In [47]:
class SigmoidFuzzyQuantizer:
    """Binary (K=2) fuzzy quantizer, Eq. (3). c=0, tau->0 recovers crisp LPQ."""
    def __init__(self, n_coeffs):
        self.c = np.zeros(n_coeffs)
        self.tau = np.ones(n_coeffs)

    def membership(self, coeffs):
        c = self.c.reshape(-1, 1, 1)
        tau = self.tau.reshape(-1, 1, 1)
        return 1.0 / (1.0 + np.exp(-(coeffs - c) / np.maximum(tau, 1e-6)))


class GaussianRBFPartition:
    """K-level fuzzy partition, Eq. (4)."""
    def __init__(self, n_coeffs, K):
        self.K = K
        self.centers = np.zeros((n_coeffs, K))
        self.sigmas = np.ones((n_coeffs, K))

    def membership(self, coeffs):
        n, H, W = coeffs.shape
        mus = np.zeros((n, self.K, H, W))
        for k in range(n):
            c = self.centers[k].reshape(self.K, 1, 1)
            s = self.sigmas[k].reshape(self.K, 1, 1)
            neg_d2 = -((coeffs[k][None] - c) ** 2) / (2 * s ** 2 + 1e-12)
            neg_d2 = neg_d2 - neg_d2.max(axis=0, keepdims=True)  # log-sum-exp stability
            unnorm = np.exp(neg_d2)
            mus[k] = unnorm / (unnorm.sum(axis=0, keepdims=True) + 1e-12)
        return mus


## 4. Adaptive fuzzy partition learning — Algorithm 3

An EM-style gradient-on-center procedure: the **E-step** computes soft assignments of
pooled training coefficients to the current partition; the **M-step** re-estimates each
center as the membership-weighted mean, and gradient-ascends the bandwidth/temperature
toward a target partition entropy (a proxy for the entropy-vs-margin trade-off of the
paper), clipped to avoid degeneracy or collapse to a uniform partition.


In [48]:
class AdaptiveFuzzyPartitionLearner:
    def __init__(self, K=2, lr=60, max_iter=30, sigma_min=1e-4, tol=1e-5):
        self.K = K
        self.lr = lr
        self.max_iter = max_iter
        self.sigma_min = sigma_min
        self.tol = tol

    def fit_one_coefficient(self, samples):
        samples = samples.ravel()
        if self.K == 2:
            scale = np.std(samples) + 1e-6
            c = np.median(samples)
            tau = 0.5 * scale
            target_entropy = 0.85  # normalized binary entropy target in (0,1]
            tau_min, tau_max = 1e-3 * scale, 3.0 * scale
            for _ in range(self.max_iter):
                mu = 1.0 / (1.0 + np.exp(-(samples - c) / max(tau, 1e-6)))
                new_c = np.average(samples, weights=mu * (1 - mu) + 1e-6)
                entropy = np.mean(-(mu * np.log(mu + 1e-12) + (1 - mu) * np.log(1 - mu + 1e-12))) / np.log(2)
                grad = target_entropy - entropy
                new_tau = np.clip(tau * (1.0 + self.lr * grad), tau_min, tau_max)
                shift = abs(new_c - c) + abs(new_tau - tau)
                c, tau = new_c, new_tau
                if shift < self.tol:
                    break
            return {"c": c, "tau": tau}
        else:
            quantiles = np.linspace(0, 1, self.K + 2)[1:-1]
            centers = np.quantile(samples, quantiles)
            spread = (samples.max() - samples.min()) / (self.K + 1) + 1e-6
            sigmas = np.full(self.K, spread)
            target_entropy = 0.5 * np.log(self.K)
            for _ in range(self.max_iter):
                d2 = (samples[None, :] - centers[:, None]) ** 2
                unnorm = np.exp(-d2 / (2 * sigmas[:, None] ** 2 + 1e-12))
                mu = unnorm / (unnorm.sum(axis=0, keepdims=True) + 1e-12)
                new_centers = (mu * samples[None, :]).sum(axis=1) / (mu.sum(axis=1) + 1e-12)
                entropy = np.mean(-(mu * np.log(mu + 1e-12)).sum(axis=0))
                grad = target_entropy - entropy
                new_sigmas = np.clip(sigmas + self.lr * grad * sigmas, self.sigma_min, None)
                shift = np.max(np.abs(new_centers - centers))
                centers, sigmas = new_centers, new_sigmas
                if shift < self.tol:
                    break
            return {"centers": centers, "sigmas": sigmas}

    def fit(self, coeff_stack):
        """coeff_stack: (n, num_samples) pooled scalar coefficients per coefficient index."""
        n = coeff_stack.shape[0]
        params = [self.fit_one_coefficient(coeff_stack[k]) for k in range(n)]
        if self.K == 2:
            quantizer = SigmoidFuzzyQuantizer(n)
            quantizer.c = np.array([p["c"] for p in params])
            quantizer.tau = np.array([p["tau"] for p in params])
            return quantizer
        else:
            part = GaussianRBFPartition(n, self.K)
            part.centers = np.stack([p["centers"] for p in params])
            part.sigmas = np.stack([p["sigmas"] for p in params])
            return part


## 5. Fuzzy soft-histogram extraction — Algorithms 4-5 (general $K \ge 2$)

For the binary case ($K=2$), the per-pixel vote weight vector over all $2^n$ candidate
codes is built by successive Kronecker (outer) products of $[1-\mu_k, \mu_k]$ across bits
(Algorithm 5, exact joint-code histogram), vectorized here over all pixels of a grid
cell at once.

For the general $K$-level case, the analogous joint-code histogram would require
$K^n$ bins (e.g. $4^{12} \approx 1.7\times10^7$ for LPQ$+$ with $K=4$), which is not
computationally tractable in general. We therefore use a **marginal soft histogram**
for $K>2$: for each coefficient $k$ we keep the $K$-bin histogram of its own average
membership across the cell (dimension $n\times K$ per grid cell, instead of $K^n$) --
still a valid soft aggregation of the fuzzy descriptor, at a linear rather than
exponential cost in $n$. The exact joint-code histogram of Algorithm 5 remains
available (and is used automatically) whenever $K=2$. In both cases, the raw
per-pixel fuzzy descriptor (`local` output) is also exposed, unchanged, for Fisher
Vector encoding (Section 7), which is exact and combinatorially unaffected by $K$
since it operates directly on the continuous membership vectors rather than on a
discretized code histogram.


In [49]:
def fuzzy_soft_histogram_binary(memberships, grid=(4, 4)):
    """Exact joint-code soft histogram (Algorithm 5), K=2 case.
    memberships: (n, H, W) in [0,1]."""
    n, H, W = memberships.shape
    hists = []
    for sy, sx in _grid_slices(H, W, grid):
        flat = memberships[:, sy, sx].reshape(n, -1)
        w = np.stack([1 - flat[0], flat[0]], axis=0)
        for k in range(1, n):
            mk = flat[k]
            w = np.concatenate([w * (1 - mk)[None, :], w * mk[None, :]], axis=0)
        h = w.sum(axis=1)
        h = h / (h.sum() + 1e-12)
        hists.append(h)
    return np.concatenate(hists)


def fuzzy_marginal_histogram(memberships_K, grid=(4, 4)):
    """Tractable soft-histogram approximation for the general K>2 case.
    memberships_K: (n, K, H, W) in [0,1], each memberships_K[k] summing to 1 over K.
    Returns, per grid cell, the concatenation of the K-bin average membership
    histogram for each of the n coefficients (dimension n*K per cell)."""
    n, K, H, W = memberships_K.shape
    hists = []
    for sy, sx in _grid_slices(H, W, grid):
        cell = memberships_K[:, :, sy, sx].reshape(n, K, -1)
        h = cell.mean(axis=2)  # (n, K) average membership per level, per coefficient
        h = h / (h.sum(axis=1, keepdims=True) + 1e-12)
        hists.append(h.ravel())
    return np.concatenate(hists)


def fuzzy_local_descriptor(memberships):
    """Per-pixel raw fuzzy descriptor vector, for Fisher Vector encoding.
    Binary case (n, H, W) -> shape (H*W, n).
    General K-level case (n, K, H, W) -> shape (H*W, n*K) (memberships flattened
    across coefficients and levels)."""
    if memberships.ndim == 3:
        n, H, W = memberships.shape
        return memberships.reshape(n, -1).T
    else:
        n, K, H, W = memberships.shape
        return memberships.reshape(n * K, -1).T


def extract_fuzzy_descriptor(image, configs, quantizers, grid=(4, 4)):
    """Works transparently for both binary (SigmoidFuzzyQuantizer, K=2) and
    general (GaussianRBFPartition, K>2) quantizers, dispatching on the shape of the
    computed membership array."""
    hist_parts, local_parts = [], []
    for cfg, quant in zip(configs, quantizers):
        coeffs = stft_coefficients(image, **cfg)
        mu = quant.membership(coeffs)
        if mu.ndim == 3:  # K=2: exact joint-code soft histogram
            hist_parts.append(fuzzy_soft_histogram_binary(mu, grid=grid))
        else:  # K>2: tractable marginal soft histogram
            hist_parts.append(fuzzy_marginal_histogram(mu, grid=grid))
        local_parts.append(fuzzy_local_descriptor(mu))
    return np.concatenate(hist_parts), np.concatenate(local_parts, axis=1)


## 6. Test blurring protocol (Gaussian / motion / circular, matching `fspecial`)

Following the paper's blur degradation protocol, three blur types are applied to the
test partitions, each at the following parameter settings:

* **Gaussian blurring** — neighborhood window size $3\times3$, standard deviation
  $\sigma \in \{1, 2, 3\}$.
* **Motion blurring** — line motion length $\lambda \in \{8, 9\}$ (camera motion),
  angle $= 0^\circ$.
* **Circular (disk) blurring** — radius $\gamma \in \{2, 3\}$.

This gives $3 + 2 + 2 = 7$ distinct blur-degraded test conditions, each generated to
match MATLAB's `fspecial` conventions for direct comparability with the reference
protocols of Xiao et al. (2017) and Zhu et al. (2021).


In [50]:
def fspecial_gaussian(hsize=3, sigma=1.0):
    r = hsize // 2
    ys = np.arange(-r, r + 1)
    Y1, Y2 = np.meshgrid(ys, ys, indexing='ij')
    h = np.exp(-(Y1 ** 2 + Y2 ** 2) / (2 * sigma ** 2))
    return h / h.sum()


def fspecial_disk(radius=3):
    r = int(np.ceil(radius))
    ys = np.arange(-r, r + 1)
    Y1, Y2 = np.meshgrid(ys, ys, indexing='ij')
    mask = (Y1 ** 2 + Y2 ** 2) <= radius ** 2
    h = mask.astype(np.float64)
    return h / h.sum()


def fspecial_motion(length=8.0, angle=0.0):
    """Line motion-blur kernel of the given length (camera motion) and angle (degrees),
    matching MATLAB's fspecial('motion', len, theta) convention."""
    length = max(int(round(length)), 1)
    theta = np.deg2rad(angle)
    half = (length - 1.0) / 2.0
    size = int(2 * np.ceil(half) + 1)
    h = np.zeros((size, size))
    center = size // 2
    for t in np.linspace(-half, half, max(length * 4, 2)):
        x = int(round(center + t * np.cos(theta)))
        y = int(round(center - t * np.sin(theta)))
        if 0 <= y < size and 0 <= x < size:
            h[y, x] = 1.0
    if h.sum() == 0:
        h[center, center] = 1.0
    return h / h.sum()


def apply_blur(image, kernel):
    return fftconvolve(image.astype(np.float64), kernel, mode='same')


# --- Test blurring protocol: Gaussian (3x3, sigma=1,2,3), motion (len=8,9, angle=0),
#     circular / disk (radius=2,3) --------------------------------------------------
BLUR_BANK = {
    "gaussian_3x3_sigma1": lambda: fspecial_gaussian(hsize=3, sigma=1),
    "gaussian_3x3_sigma2": lambda: fspecial_gaussian(hsize=3, sigma=2),
    "gaussian_3x3_sigma3": lambda: fspecial_gaussian(hsize=3, sigma=3),
    "motion_len8_angle0": lambda: fspecial_motion(length=8.0, angle=0),
    "motion_len9_angle0": lambda: fspecial_motion(length=9.0, angle=0),
    "disk_radius2": lambda: fspecial_disk(radius=2),
    "disk_radius3": lambda: fspecial_disk(radius=3),
}


In [ ]:
# Visualize the effect of every blur condition on a sample texture patch.
_sample = 128 + 100 * np.sin(2 * np.pi * np.tile(np.arange(48), (48, 1)) / 6)
_sample = np.clip(_sample + np.random.RandomState(0).randn(48, 48) * 8, 0, 255)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.ravel()
axes[0].imshow(_sample, cmap='gray'); axes[0].set_title('original'); axes[0].axis('off')
for ax, (name, kfn) in zip(axes[1:], BLUR_BANK.items()):
    blurred = apply_blur(_sample, kfn())
    ax.imshow(blurred, cmap='gray')
    ax.set_title(name, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()


## 7. Fisher Vector encoding — Algorithms 6-7

A diagonal-covariance GMM is trained (offline, via EM) on pooled per-pixel fuzzy
descriptors from the training set (`train_fv_gmm`), then each image's descriptor set is
encoded into a mean/variance-deviation Fisher Vector, power- and $\ell_2$-normalized
(`fisher_vector_encode`).


In [32]:
def train_fv_gmm(pooled_local_descriptors, n_components=32, seed=0):
    gmm = GaussianMixture(n_components=n_components, covariance_type='diag',
                           random_state=seed, reg_covar=1e-4, max_iter=150)
    gmm.fit(pooled_local_descriptors)
    return gmm


def fisher_vector_encode(local_descriptors, gmm):
    T, d = local_descriptors.shape
    means = gmm.means_
    covs = gmm.covariances_
    weights = gmm.weights_
    M = means.shape[0]
    gamma = gmm.predict_proba(local_descriptors)  # (T, M)
    G_X = np.zeros((M, d))
    G_S = np.zeros((M, d))
    for m in range(M):
        diff = (local_descriptors - means[m]) / np.sqrt(covs[m] + 1e-12)
        g = gamma[:, m][:, None]
        G_X[m] = (g * diff).sum(axis=0) / (T * np.sqrt(weights[m]) + 1e-12)
        G_S[m] = (g * (diff ** 2 - 1)).sum(axis=0) / (T * np.sqrt(2 * weights[m]) + 1e-12)
    fv = np.concatenate([G_X.ravel(), G_S.ravel()])
    fv = np.sign(fv) * np.sqrt(np.abs(fv))
    fv = fv / (np.linalg.norm(fv) + 1e-12)
    return fv


## 8. End-to-end pipeline — Algorithm 8

`load_dataset` expects a folder structured as `root/<class_name>/<image files>` (this is
the layout you'd get from unzipping KTH-TIPS, Outex or Kylberg into per-class
subfolders on Google Drive). `run_experiment` performs the full pipeline for one
`(descriptor, mode)` combination: `descriptor` &isin; {"LPQ","LPQ6","MP-LPQ"},
`mode` &isin; {"crisp_hist","crisp_fv","fuzzy_hist","fuzzy_fv"}. The `crisp_fv` mode
embeds the *crisp* (hard 0/1) local bit descriptors into a Fisher Vector, giving a
clean ablation of "does FV help on its own" independent of fuzzification -- directly
comparable to `fuzzy_fv`, which embeds the *fuzzy* (soft-membership) local descriptors
instead.


In [52]:
import os
from glob import glob
try:
    from PIL import Image
except ImportError:
    Image = None


def load_dataset(root_dir, extensions=(".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"),
                  resize=None, max_per_class=None):
    classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    images, labels = [], []
    for cls in classes:
        paths = []
        for ext in extensions:
            paths += glob(os.path.join(root_dir, cls, f"*{ext}"))
        paths = sorted(paths)
        if max_per_class is not None:
            paths = paths[:max_per_class]
        for p in paths:
            img = Image.open(p).convert("L")
            if resize is not None:
                img = img.resize(resize)
            images.append(np.array(img, dtype=np.float64))
            labels.append(cls)
    return images, labels, classes


def fit_fuzzy_quantizers(images, descriptor="LPQ", K=2, learner_kwargs=None):
    """Pool STFT coefficients across training images and learn one fuzzy partition
    per coefficient index, per frequency-point configuration (Algorithm 3)."""
    configs = CONFIG_MAP[descriptor]
    learner_kwargs = learner_kwargs or {}
    quantizers = []
    for cfg in configs:
        pooled = []
        for img in images:
            coeffs = stft_coefficients(img, **cfg)
            pooled.append(coeffs.reshape(coeffs.shape[0], -1))
        pooled = np.concatenate(pooled, axis=1)
        learner = AdaptiveFuzzyPartitionLearner(K=K, **learner_kwargs)
        quantizers.append(learner.fit(pooled))
    return configs, quantizers


def batch_extract(images, mode, descriptor="LPQ", grid=(4, 4), quantizers=None, gmm=None):
    """
    mode: 'crisp_hist' | 'crisp_fv' | 'fuzzy_hist' | 'fuzzy_fv'
    Returns (feature_matrix, local_descriptors_per_image); the latter is only populated
    (as a list) when mode in {'crisp_fv','fuzzy_fv'} and gmm is None (pooling stage).
    """
    configs = CONFIG_MAP[descriptor]
    feats, locals_ = [], []
    for img in images:
        if mode in ("crisp_hist", "crisp_fv"):
            hist, local = extract_crisp_descriptor_with_local(img, configs, grid=grid)
            if mode == "crisp_hist":
                feats.append(hist)
            else:
                locals_.append(local)
                if gmm is not None:
                    feats.append(fisher_vector_encode(local, gmm))
        elif mode in ("fuzzy_hist", "fuzzy_fv"):
            hist, local = extract_fuzzy_descriptor(img, configs, quantizers, grid=grid)
            if mode == "fuzzy_hist":
                feats.append(hist)
            else:
                locals_.append(local)
                if gmm is not None:
                    feats.append(fisher_vector_encode(local, gmm))
        else:
            raise ValueError(f"unknown mode {mode}")
    if mode in ("crisp_fv", "fuzzy_fv") and gmm is None:
        return None, locals_
    return np.stack(feats), locals_


def run_experiment(train_images, train_labels, test_images, test_labels,
                    descriptor="LPQ", mode="crisp_hist", K=2, grid=(4, 4),
                    n_gaussians=32, seed=0, svm_kernel="rbf", svm_C=10.0):
    """Full Algorithm 8 pipeline for one (descriptor, mode) combination."""
    quantizers, gmm = None, None

    if mode in ("fuzzy_hist", "fuzzy_fv"):
        _, quantizers = fit_fuzzy_quantizers(train_images, descriptor=descriptor, K=K)

    if mode == "crisp_hist":
        Xtr, _ = batch_extract(train_images, "crisp_hist", descriptor, grid)
        Xte, _ = batch_extract(test_images, "crisp_hist", descriptor, grid)
    elif mode == "crisp_fv":
        _, local_tr = batch_extract(train_images, "crisp_fv", descriptor, grid, gmm=None)
        pooled = np.concatenate(local_tr, axis=0)
        if pooled.shape[0] > 20000:
            idx = np.random.RandomState(seed).choice(pooled.shape[0], 20000, replace=False)
            pooled = pooled[idx]
        gmm = train_fv_gmm(pooled, n_components=n_gaussians, seed=seed)
        Xtr, _ = batch_extract(train_images, "crisp_fv", descriptor, grid, gmm=gmm)
        Xte, _ = batch_extract(test_images, "crisp_fv", descriptor, grid, gmm=gmm)
    elif mode == "fuzzy_hist":
        Xtr, _ = batch_extract(train_images, "fuzzy_hist", descriptor, grid, quantizers)
        Xte, _ = batch_extract(test_images, "fuzzy_hist", descriptor, grid, quantizers)
    elif mode == "fuzzy_fv":
        _, local_tr = batch_extract(train_images, "fuzzy_fv", descriptor, grid, quantizers, gmm=None)
        pooled = np.concatenate(local_tr, axis=0)
        if pooled.shape[0] > 20000:
            idx = np.random.RandomState(seed).choice(pooled.shape[0], 20000, replace=False)
            pooled = pooled[idx]
        gmm = train_fv_gmm(pooled, n_components=n_gaussians, seed=seed)
        Xtr, _ = batch_extract(train_images, "fuzzy_fv", descriptor, grid, quantizers, gmm=gmm)
        Xte, _ = batch_extract(test_images, "fuzzy_fv", descriptor, grid, quantizers, gmm=gmm)
    else:
        raise ValueError(mode)

    clf = SVC(kernel=svm_kernel, C=svm_C,gamma='scale')
    clf.fit(Xtr, train_labels)
    preds = clf.predict(Xte)
    acc = accuracy_score(test_labels, preds)
    report = classification_report(test_labels, preds, zero_division=0)
    return acc, report, (Xtr, Xte)


def build_blur_test_sets(images, labels):
    """Apply every condition of the test blurring protocol (Section 6) to a set of
    clean test images. Returns {'no_blur': (images, labels), <blur_name>: (blurred, labels), ...}."""
    out = {"no_blur": (images, labels)}
    for name, kernel_fn in BLUR_BANK.items():
        kernel = kernel_fn()
        out[name] = ([apply_blur(im, kernel) for im in images], labels)
    return out


def fit_texture_model(train_images, train_labels, descriptor="LPQ", mode="crisp_hist",
                       K=2, grid=(4, 4), n_gaussians=32, seed=0, svm_kernel="rbf", svm_C=10.0):
    """Fit fuzzy quantizers (if applicable), GMM (if applicable) and the classifier ONCE
    on clean training images, so the same trained model can be evaluated across every
    blur condition without refitting each time."""
    quantizers, gmm = None, None
    if mode in ("fuzzy_hist", "fuzzy_fv"):
        _, quantizers = fit_fuzzy_quantizers(train_images, descriptor=descriptor, K=K)

    if mode == "crisp_hist":
        Xtr, _ = batch_extract(train_images, "crisp_hist", descriptor, grid)
    elif mode == "crisp_fv":
        _, local_tr = batch_extract(train_images, "crisp_fv", descriptor, grid, gmm=None)
        pooled = np.concatenate(local_tr, axis=0)
        if pooled.shape[0] > 20000:
            idx = np.random.RandomState(seed).choice(pooled.shape[0], 20000, replace=False)
            pooled = pooled[idx]
        gmm = train_fv_gmm(pooled, n_components=n_gaussians, seed=seed)
        Xtr, _ = batch_extract(train_images, "crisp_fv", descriptor, grid, gmm=gmm)
    elif mode == "fuzzy_hist":
        Xtr, _ = batch_extract(train_images, "fuzzy_hist", descriptor, grid, quantizers)
    elif mode == "fuzzy_fv":
        _, local_tr = batch_extract(train_images, "fuzzy_fv", descriptor, grid, quantizers, gmm=None)
        pooled = np.concatenate(local_tr, axis=0)
        if pooled.shape[0] > 20000:
            idx = np.random.RandomState(seed).choice(pooled.shape[0], 20000, replace=False)
            pooled = pooled[idx]
        gmm = train_fv_gmm(pooled, n_components=n_gaussians, seed=seed)
        Xtr, _ = batch_extract(train_images, "fuzzy_fv", descriptor, grid, quantizers, gmm=gmm)
    else:
        raise ValueError(mode)

    clf = SVC(kernel=svm_kernel, C=svm_C)
    clf.fit(Xtr, train_labels)
    return {"descriptor": descriptor, "mode": mode, "grid": grid,
            "quantizers": quantizers, "gmm": gmm, "clf": clf}


def evaluate_texture_model(model, test_images, test_labels):
    Xte, _ = batch_extract(test_images, model["mode"], model["descriptor"], model["grid"],
                            model["quantizers"], model["gmm"])
    preds = model["clf"].predict(Xte)
    return accuracy_score(test_labels, preds)


## 9a. Demo on synthetic textures (sanity check, runs instantly)

This cell fabricates a few toy texture classes so the whole pipeline can be exercised
without needing to download a dataset first. Replace this with the real Google Drive
loader in the next cell for actual KTH-TIPS / Outex / Kylberg experiments. Note that
training always uses **clean** images, while the **test** partition is evaluated once
per condition of the test blurring protocol from Section 6 (Gaussian $\sigma=1,2,3$;
motion $\lambda=8,9$ at $0^\circ$; disk $\gamma=2,3$), plus a clean ("no_blur")
baseline — exactly mirroring the paper's evaluation protocol.


In [ ]:
def make_texture(kind, size=48, seed=None):
    rng = np.random.RandomState(seed)
    ys, xs = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
    if kind == 'stripes':
        img = 128 + 100 * np.sin(2 * np.pi * xs / 6)
    elif kind == 'checker':
        img = 128 + 100 * np.sign(np.sin(2 * np.pi * xs / 8) * np.sin(2 * np.pi * ys / 8))
    elif kind == 'waves':
        img = 128 + 100 * np.sin(2 * np.pi * xs / 5) * np.cos(2 * np.pi * ys / 5)
    else:
        raise ValueError(kind)
    noise = rng.randn(size, size) * 10
    return np.clip(img + noise, 0, 255)


classes = ['stripes', 'checker', 'waves']
train_images, train_labels = [], []
test_images_clean, test_labels = [], []
for c in classes:
    for i in range(8):
        train_images.append(make_texture(c, size=32, seed=i))
        train_labels.append(c)
    for i in range(4):
        test_images_clean.append(make_texture(c, size=32, seed=1000 + i))
        test_labels.append(c)

blur_test_sets = build_blur_test_sets(test_images_clean, test_labels)
print(f"{len(train_images)} clean training images, {len(test_images_clean)} test images "
      f"x {len(blur_test_sets)} conditions ({', '.join(blur_test_sets.keys())})")


In [ ]:
# Fit each (descriptor, mode) combination ONCE on clean training data, then evaluate it
# against every blur condition (no refitting per condition).
descriptors = ["LPQ", "LPQ6", "MP-LPQ"]
modes = ["crisp_hist", "crisp_fv", "fuzzy_hist", "fuzzy_fv"]
blur_names = list(blur_test_sets.keys())

full_results = {}  # (descriptor, mode) -> {blur_name: acc}
for descriptor in descriptors:
    for mode in modes:
        model = fit_texture_model(train_images, train_labels, descriptor=descriptor,
                                   mode=mode, grid=(2, 2), n_gaussians=8)
        row = {}
        for blur_name, (imgs, labs) in blur_test_sets.items():
            row[blur_name] = evaluate_texture_model(model, imgs, labs)
        full_results[(descriptor, mode)] = row
        print(f"{descriptor:6s} | {mode:11s} | mean acc over all blur conditions = "
              f"{np.mean(list(row.values())):.3f}")


In [ ]:
# Summary table 1: mean accuracy over all blur conditions, now comparing FV embedding
# for BOTH the crisp descriptor (crisp_fv) and the fuzzy descriptor (fuzzy_fv) -- this
# separates "does FV help" from "does fuzzification help" as two independent axes.
print(f"{'Descriptor':10s} {'crisp_hist':>11s} {'crisp_fv':>10s} {'fuzzy_hist':>11s} {'fuzzy_fv':>10s}")
for descriptor in descriptors:
    means = {mode: np.mean(list(full_results[(descriptor, mode)].values())) for mode in modes}
    print(f"{descriptor:10s} {means['crisp_hist']:11.3f} {means['crisp_fv']:10.3f} "
          f"{means['fuzzy_hist']:11.3f} {means['fuzzy_fv']:10.3f}")

print()
# Summary table 2: per-blur-condition breakdown for the best-performing variant
# (Fuzzy MP-LPQ + FV), mirroring the blur-robustness ablation of the paper.
best = ("MP-LPQ", "fuzzy_fv")
print(f"Per-condition accuracy for {best[0]} + {best[1]}:")
for blur_name in blur_names:
    print(f"  {blur_name:22s} {full_results[best][blur_name]:.3f}")


## 9c. Beyond binary: the general $K>2$ case (Gaussian RBF fuzzy partition)

Everything above used the binary sigmoid quantizer ($K=2$). Every function in this
notebook (`AdaptiveFuzzyPartitionLearner`, `extract_fuzzy_descriptor`,
`fit_fuzzy_quantizers`, `fit_texture_model`, `run_experiment`,
`cross_validate_all_methods`) accepts a `K` argument and dispatches automatically:

* `K=2` &rarr; `SigmoidFuzzyQuantizer`, exact joint-code soft histogram (Algorithm 5).
* `K>2` &rarr; `GaussianRBFPartition`, marginal soft histogram (Section 5) for
  `fuzzy_hist`, and an exact (combinatorially unaffected) Fisher Vector encoding for
  `fuzzy_fv`, since FV operates on the raw $n\times K$-dimensional membership vector
  rather than on a discretized code histogram.

The cell below repeats the MP-LPQ comparison for $K \in \{2, 4\}$ (extend this list,
e.g. to $\{2,3,4,5\}$, for a finer-grained sweep on a real dataset), for both
`fuzzy_hist` and `fuzzy_fv`, so you can see how the number of fuzzy partition levels
trades off against accuracy (too few levels under-utilizes the continuous phase
information; too many can fragment the partition and overfit on small training sets).


In [ ]:
K_values = [2, 4]
k_results = {}  # (K, mode) -> mean accuracy over all blur conditions
for K in K_values:
    for mode in ["fuzzy_hist", "fuzzy_fv"]:
        model = fit_texture_model(train_images, train_labels, descriptor="LPQ++",
                                   mode=mode, K=K, grid=(2, 2), n_gaussians=8)
        accs = [evaluate_texture_model(model, imgs, labs)
                for imgs, labs in blur_test_sets.values()]
        k_results[(K, mode)] = np.mean(accs)

print(f"{'K':>3s} {'fuzzy_hist':>12s} {'fuzzy_fv':>12s}")
for K in K_values:
    print(f"{K:3d} {k_results[(K, 'fuzzy_hist')]:12.3f} {k_results[(K, 'fuzzy_fv')]:12.3f}")


## 9b. Running on real datasets (KTH-TIPS / Outex / Kylberg)

1. Unzip your dataset on Google Drive so that each class is its own subfolder:
   ```
   /content/drive/MyDrive/textures/kth_tips/<class_name>/*.png
   ```
2. Load it, split into train/test (e.g. standard splits from Xiao et al. 2017 /
   Zhu et al. 2021, or a random stratified split for a quick check), optionally
   apply the blur bank to the test partition, then call `run_experiment` exactly as
   in the demo above.


In [ ]:
# Example (uncomment and adapt the path once your dataset is mounted):
import warnings
warnings.filterwarnings('ignore')
images, labels, classes = load_dataset(
    "/content/Data/yale_db",
    resize=(128, 128), max_per_class=60,
)

from sklearn.model_selection import train_test_split
train_images, test_images_clean, train_labels, test_labels = train_test_split(
    images, labels, test_size=0.67, stratify=labels, random_state=30,
)


print(len(train_images))
# Build the full test blurring protocol (Section 6) from the clean test images:
blur_test_sets = build_blur_test_sets(test_images_clean, test_labels)

# Fit once on clean training data, evaluate across every blur condition:
print("=================== Mode1: LPQ_HIST_CRSIP =====================")
model = fit_texture_model(train_images, train_labels, descriptor="LPQ",
                           mode="crisp_hist", grid=(2, 2), n_gaussians=32)
for blur_name, (imgs, labs) in blur_test_sets.items():
    acc = evaluate_texture_model(model, imgs, labs)
    print(f"{blur_name:22s} {acc:.3f}")
print("================== Mode2: LPQ_HIST_FUZZY =====================")

model = fit_texture_model(train_images, train_labels, descriptor="LPQ",
                           mode="fuzzy_hist", grid=(2, 2), n_gaussians=32)
for blur_name, (imgs, labs) in blur_test_sets.items():
    acc = evaluate_texture_model(model, imgs, labs)
    print(f"{blur_name:22s} {acc:.3f}")
print("================== Mode3: LPQ_FV_CRISP =======================")
model = fit_texture_model(train_images, train_labels, descriptor="LPQ",
                           mode="crisp_fv", grid=(2, 2), n_gaussians=5)
for blur_name, (imgs, labs) in blur_test_sets.items():
    acc = evaluate_texture_model(model, imgs, labs)
    print(f"{blur_name:22s} {acc:.3f}")
print("================== Mode4: LPQ_FV_FUZZY ========================")

model = fit_texture_model(train_images, train_labels, descriptor="LPQ",
                           mode="fuzzy_fv", grid=(2, 2), n_gaussians=5)
for blur_name, (imgs, labs) in blur_test_sets.items():
    acc = evaluate_texture_model(model, imgs, labs)
    print(f"{blur_name:22s} {acc:.3f}")





In [ ]:
# @title Evaluation of LPQ
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

images, labels, classes = load_dataset(
    "/content/Data/yale_db",
    resize=(128, 128),
    max_per_class=11,
)

# Définition des modes à évaluer
modes_config = [
    ("Mode1: LPQ_HIST_CRISP", dict(descriptor="LPQ", mode="crisp_hist", grid=(4, 4), n_gaussians=32)),
    ("Mode2: LPQ_HIST_FUZZY", dict(descriptor="LPQ", mode="fuzzy_hist", grid=(4, 4), n_gaussians=32)),
    ("Mode3: LPQ_FV_CRISP",   dict(descriptor="LPQ", mode="crisp_fv",   grid=(4, 4), n_gaussians=32)),
    ("Mode4: LPQ_FV_FUZZY",   dict(descriptor="LPQ", mode="fuzzy_fv",   grid=(4, 4), n_gaussians=32)),
]

n_splits = 5
sss = StratifiedShuffleSplit(
    n_splits=n_splits,
    train_size=4/11,
    test_size=7/11,
    random_state=30,
)

labels = np.array(labels)
images = np.array(images, dtype=object) if not isinstance(images, np.ndarray) else images

# results[mode_name][blur_name] = liste des accuracies sur les 5 tirages
results = {name: {} for name, _ in modes_config}

for fold_idx, (train_idx, test_idx) in enumerate(sss.split(images, labels), start=1):
    print(f"\n########## FOLD {fold_idx}/{n_splits} (train=33%, test=67%) ##########")

    train_images = [images[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    test_images_clean = [images[i] for i in test_idx]
    test_labels = [labels[i] for i in test_idx]

    print(f"train={len(train_images)}  test={len(test_images_clean)}")

    # Reconstruire le protocole de flou (Section 6) sur le test set de ce fold
    blur_test_sets = build_blur_test_sets(test_images_clean, test_labels)

    for mode_name, params in modes_config:
        print(f"=== {mode_name} (fold {fold_idx}) ===")
        model = fit_texture_model(train_images, train_labels, **params)

        for blur_name, (imgs, labs) in blur_test_sets.items():
            acc = evaluate_texture_model(model, imgs, labs)
            print(f"{blur_name:22s} {acc:.3f}")
            results[mode_name].setdefault(blur_name, []).append(acc)

# ===================== Rapport final : moyenne +/- écart-type sur les 5 tirages =====================
print("\n\n===================== RESULTATS MOYENS (5x train=33%/test=67%) =====================")
for mode_name, blur_dict in results.items():
    print(f"\n--- {mode_name} ---")
    for blur_name, accs in blur_dict.items():
        accs = np.array(accs)
        print(f"{blur_name:22s} mean={accs.mean():.3f}  std={accs.std():.3f}  (n={len(accs)})")

In [ ]:
# @title Evaluation of LPQ6
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

images, labels, classes = load_dataset(
    "/content/Data/yale_db",
    resize=(128, 128),
    max_per_class=11,
)

# Définition des modes à évaluer
modes_config = [
    ("Mode1: LPQ6_HIST_CRISP", dict(descriptor="LPQ6", mode="crisp_hist", grid=(4, 4), n_gaussians=32)),
    ("Mode2: LPQ6_HIST_FUZZY", dict(descriptor="LPQ6", mode="fuzzy_hist", grid=(4, 4), n_gaussians=32)),
    ("Mode3: LPQ6_FV_CRISP",   dict(descriptor="LPQ6", mode="crisp_fv",   grid=(4, 4), n_gaussians=32)),
    ("Mode4: LPQ6_FV_FUZZY",   dict(descriptor="LPQ6", mode="fuzzy_fv",   grid=(4, 4), n_gaussians=32)),
]

n_splits = 5
sss = StratifiedShuffleSplit(
    n_splits=n_splits,
    train_size=4/11,
    test_size=7/11,
    random_state=30,
)

labels = np.array(labels)
images = np.array(images, dtype=object) if not isinstance(images, np.ndarray) else images

# results[mode_name][blur_name] = liste des accuracies sur les 5 tirages
results = {name: {} for name, _ in modes_config}

for fold_idx, (train_idx, test_idx) in enumerate(sss.split(images, labels), start=1):
    print(f"\n########## FOLD {fold_idx}/{n_splits} (train=33%, test=67%) ##########")

    train_images = [images[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    test_images_clean = [images[i] for i in test_idx]
    test_labels = [labels[i] for i in test_idx]

    print(f"train={len(train_images)}  test={len(test_images_clean)}")

    # Reconstruire le protocole de flou (Section 6) sur le test set de ce fold
    blur_test_sets = build_blur_test_sets(test_images_clean, test_labels)

    for mode_name, params in modes_config:
        print(f"=== {mode_name} (fold {fold_idx}) ===")
        model = fit_texture_model(train_images, train_labels, **params)

        for blur_name, (imgs, labs) in blur_test_sets.items():
            acc = evaluate_texture_model(model, imgs, labs)
            print(f"{blur_name:22s} {acc:.3f}")
            results[mode_name].setdefault(blur_name, []).append(acc)

# ===================== Rapport final : moyenne +/- écart-type sur les 5 tirages =====================
print("\n\n===================== RESULTATS MOYENS (5x train=33%/test=67%) =====================")
for mode_name, blur_dict in results.items():
    print(f"\n--- {mode_name} ---")
    for blur_name, accs in blur_dict.items():
        accs = np.array(accs)
        print(f"{blur_name:22s} mean={accs.mean():.3f}  std={accs.std():.3f}  (n={len(accs)})")

In [ ]:
# @title Evaluation of MP-LPQ
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

images, labels, classes = load_dataset(
    "/content/Data/yale_db",
    resize=(128, 128),
    max_per_class=11,
)

# Définition des modes à évaluer
modes_config = [
    ("Mode1: MP-LPQ_HIST_CRISP", dict(descriptor="MP-LPQ", mode="crisp_hist", grid=(4, 4), n_gaussians=32)),
    ("Mode2: MP-LPQ_HIST_FUZZY", dict(descriptor="MP-LPQ", mode="fuzzy_hist", grid=(4, 4), n_gaussians=32)),
    ("Mode3: MP-LPQ_FV_CRISP",   dict(descriptor="MP-LPQ", mode="crisp_fv",   grid=(4, 4), n_gaussians=32)),
    ("Mode4: MP-LPQ_FV_FUZZY",   dict(descriptor="MP-LPQ", mode="fuzzy_fv",   grid=(4, 4), n_gaussians=32)),
]

n_splits = 5
sss = StratifiedShuffleSplit(
    n_splits=n_splits,
    train_size=4/11,
    test_size=7/11,
    random_state=30,
)

labels = np.array(labels)
images = np.array(images, dtype=object) if not isinstance(images, np.ndarray) else images

# results[mode_name][blur_name] = liste des accuracies sur les 5 tirages
results = {name: {} for name, _ in modes_config}

for fold_idx, (train_idx, test_idx) in enumerate(sss.split(images, labels), start=1):
    print(f"\n########## FOLD {fold_idx}/{n_splits} (train=33%, test=67%) ##########")

    train_images = [images[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    test_images_clean = [images[i] for i in test_idx]
    test_labels = [labels[i] for i in test_idx]

    print(f"train={len(train_images)}  test={len(test_images_clean)}")

    # Reconstruire le protocole de flou (Section 6) sur le test set de ce fold
    blur_test_sets = build_blur_test_sets(test_images_clean, test_labels)

    for mode_name, params in modes_config:
        print(f"=== {mode_name} (fold {fold_idx}) ===")
        model = fit_texture_model(train_images, train_labels, **params)

        for blur_name, (imgs, labs) in blur_test_sets.items():
            acc = evaluate_texture_model(model, imgs, labs)
            print(f"{blur_name:22s} {acc:.3f}")
            results[mode_name].setdefault(blur_name, []).append(acc)

# ===================== Rapport final : moyenne +/- écart-type sur les 5 tirages =====================
print("\n\n===================== RESULTATS MOYENS (5x train=33%/test=67%) =====================")
for mode_name, blur_dict in results.items():
    print(f"\n--- {mode_name} ---")
    for blur_name, accs in blur_dict.items():
        accs = np.array(accs)
        print(f"{blur_name:22s} mean={accs.mean():.3f}  std={accs.std():.3f}  (n={len(accs)})")

## 10. Five-fold cross-validation across all methods

For a statistically robust comparison, every `(descriptor, mode)` combination
(LPQ / LPQ6 / MP-LPQ &times; crisp\_hist / crisp\_fv / fuzzy\_hist / fuzzy\_fv,
12 methods in total) is evaluated with **5-fold stratified cross-validation**: in each
fold, the fuzzy quantizers (if applicable), the Fisher Vector GMM (if applicable), and
the SVM classifier are all fit **only** on that fold's clean training images (never
touching the held-out fold), and evaluated on the held-out fold under every condition
of the test blurring protocol (Section 6). Reported figures are the mean $\pm$
standard deviation, across the 5 folds, of the accuracy averaged over all blur
conditions.


In [ ]:
from sklearn.model_selection import StratifiedKFold


def cross_validate_all_methods(images, labels, descriptors=("LPQ", "LPQ6", "MP-LPQ"),
                                modes=("crisp_hist", "fuzzy_hist", "fuzzy_fv"),
                                n_splits=5, K=2, grid=(2, 2), n_gaussians=8, seed=0,
                                blur_subset=None, verbose=True):
    """
    Returns a dict {(descriptor, mode): {blur_name: [acc_fold_1, ..., acc_fold_k]}}.
    K: number of fuzzy partition levels for 'fuzzy_hist'/'fuzzy_fv' modes (K=2 uses the
    sigmoid quantizer with the exact joint-code soft histogram; K>2 uses the Gaussian
    RBF partition with the marginal soft-histogram approximation -- see Section 5).
    Ignored for 'crisp_hist'/'crisp_fv'.
    blur_subset: optional list of blur condition names to restrict evaluation to
    (default: all conditions from BLUR_BANK plus 'no_blur').
    """
    images = list(images)
    labels = np.asarray(labels)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    results = {(d, m): {} for d in descriptors for m in modes}
    for fold, (train_idx, test_idx) in enumerate(skf.split(images, labels), start=1):
        train_images = [images[i] for i in train_idx]
        train_labels = labels[train_idx].tolist()
        test_images_clean = [images[i] for i in test_idx]
        test_labels = labels[test_idx].tolist()

        blur_sets = build_blur_test_sets(test_images_clean, test_labels)
        if blur_subset is not None:
            blur_sets = {k: v for k, v in blur_sets.items() if k in blur_subset}

        for d in descriptors:
            for m in modes:
                model = fit_texture_model(train_images, train_labels, descriptor=d, mode=m,
                                           K=K, grid=grid, n_gaussians=n_gaussians, seed=seed)
                for blur_name, (imgs, labs) in blur_sets.items():
                    acc = evaluate_texture_model(model, imgs, labs)
                    results[(d, m)].setdefault(blur_name, []).append(acc)
        if verbose:
            print(f"fold {fold}/{n_splits} done")
    return results


def summarize_cv_results(results):
    """For each (descriptor, mode): average accuracy across blur conditions within each
    fold, then report mean +/- std of that per-fold average across the folds."""
    summary = {}
    for key, blur_dict in results.items():
        n_folds = len(next(iter(blur_dict.values())))
        per_fold_overall = [np.mean([blur_dict[b][f] for b in blur_dict]) for f in range(n_folds)]
        summary[key] = (np.mean(per_fold_overall), np.std(per_fold_overall))
    return summary


In [ ]:
# Build one combined (unsplit) synthetic dataset for cross-validation.
# (Kept small here purely so the demo runs quickly; use blur_subset=None and more
# samples/n_gaussians for a full run on a real dataset.)
all_images, all_labels = [], []
for c in classes:
    for i in range(8):
        all_images.append(make_texture(c, size=32, seed=2000 + i))
        all_labels.append(c)
print(f"{len(all_images)} images across {len(classes)} classes for 5-fold CV")

quick_blur_subset = ["no_blur", "gaussian_3x3_sigma2", "disk_radius2"]
cv_modes = ("crisp_hist", "crisp_fv", "fuzzy_hist", "fuzzy_fv")

cv_results = cross_validate_all_methods(
    all_images, all_labels,
    descriptors=("LPQ", "LPQ6", "MP-LPQ"),
    modes=cv_modes,
    n_splits=5, K=2, grid=(2, 2), n_gaussians=4, seed=0,
    blur_subset=quick_blur_subset,
)
cv_summary = summarize_cv_results(cv_results)

print()
print(f"{'Descriptor':10s} {'crisp_hist':>14s} {'crisp_fv':>14s} {'fuzzy_hist':>14s} {'fuzzy_fv':>14s}")
for d in ("LPQ", "LPQ6", "MP-LPQ"):
    cells_ = []
    for m in cv_modes:
        mean, std = cv_summary[(d, m)]
        cells_.append(f"{mean:.3f}+/-{std:.3f}")
    print(f"{d:10s} {cells_[0]:>14s} {cells_[1]:>14s} {cells_[2]:>14s} {cells_[3]:>14s}")


### Running 5-fold CV on a real dataset

Once `load_dataset(...)` has given you `images, labels, classes`, cross-validating all
12 methods (LPQ / LPQ6 / MP-LPQ &times; crisp\_hist / crisp\_fv / fuzzy\_hist / fuzzy\_fv)
is a single call; pass `K=4` (or any $K>2$) to cross-validate the general Gaussian RBF
fuzzy partition instead of the default binary sigmoid quantizer:


In [ ]:
images, labels, classes = load_dataset(
    "/content/Data/yale_db", resize=(128, 128), max_per_class=60,
)

cv_results = cross_validate_all_methods(
    images, labels,
    descriptors=("LPQ","LPQ6","MP-LPQ"),
    modes=("crisp_hist", "crisp_fv", "fuzzy_hist", "fuzzy_fv"),
    n_splits=5, K=2, grid=(4, 4), n_gaussians=32, seed=0,
)
cv_summary = summarize_cv_results(cv_results)
for key, (mean, std) in cv_summary.items():
    print(key, f"{mean:.3f} +/- {std:.3f}")
# # Or, to cross-validate the K>2 (Gaussian RBF) fuzzy partition instead:
# cv_results_k4 = cross_validate_all_methods(
#     images, labels,
#     descriptors=("LPQ", "LPQ6", "MP-LPQ"),
#     modes=("fuzzy_hist", "fuzzy_fv"),
#     n_splits=5, K=4, grid=(4, 4), n_gaussians=32, seed=0,
# )



## Notes

- **Speed**: `stft_coefficients` uses `fftconvolve`, which is fast enough for
  moderate image sizes (up to a few hundred pixels per side) and dataset sizes on a
  Colab CPU runtime; for larger-scale runs consider batching images and/or moving
  the STFT step to a GPU (e.g. via CuPy or `torch.fft`).
- **Fuzziness parameter**: the target-entropy heuristics inside
  `AdaptiveFuzzyPartitionLearner` (`target_entropy = 0.85` for $K=2$,
  `target_entropy = 0.5*log(K)` for $K>2$) control how "soft" the learned partition is;
  lowering them moves the learned quantizer closer to the crisp descriptor
  ($\tau_k \to 0$), as discussed in Section 4 and the ablation of Section 8.2 of the paper.
- **Fisher Vector dimensionality**: the FV has dimension $2Md$ where $M$ is the number
  of Gaussians and $d$ the fuzzy local descriptor dimension (`n` per configuration,
  summed across the $P$ configurations for MP-LPQ); reduce `n_gaussians` if you hit
  memory limits with large datasets.
